In [ ]:
import numpy as np
import PIL
import random
from sklearn.model_selection import cross_val_score

from PIL import Image
import matplotlib.pyplot as plt
from tensorflow.keras.regularizers import l2
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Reshape, Input, Conv2D, MaxPooling2D, Flatten, Dense, GRU
from tensorflow.keras.models import Model
import seaborn as sns
import glob
import sklearn.metrics as metrics
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, classification_report, f1_score, confusion_matrix, roc_curve, roc_auc_score,auc
from tensorflow.keras import layers, models
from tensorflow.keras.layers import Input, SeparableConv2D, BatchNormalization, Activation, Dense, Flatten, MaxPooling2D
from tensorflow.keras.layers import Add
from keras.layers import BatchNormalization
from keras.layers import ELU
from tensorflow.keras.layers import ReLU
import sys
from google.colab import files
import pandas as pd
from sklearn.preprocessing import label_binarize
import time
import numpy as np
import PIL
from PIL import Image
import matplotlib.pyplot as plt
from tensorflow.keras.regularizers import l2
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Reshape, Input, Conv2D, MaxPooling2D, Flatten,  GRU
from tensorflow.keras.models import Model
import seaborn as sns
import glob
import sklearn.metrics as metrics
from tensorflow.keras import layers, models
from tensorflow.keras.layers import Add
from keras.layers import BatchNormalization
from keras.layers import ELU
from tensorflow.keras.layers import ReLU
import sys
from google.colab import files
import pandas as pd
from sklearn.preprocessing import label_binarize
import time
from tensorflow.keras.applications import InceptionV3, ResNet50, EfficientNetB0, DenseNet201, InceptionResNetV2
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import (
    VGG16, ResNet50, MobileNetV2, EfficientNetB0, EfficientNetB3,
    DenseNet121, Xception, NASNetMobile, InceptionV3, ConvNeXtTiny
)
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import numpy as np
from tensorflow.keras.optimizers.schedules import ExponentialDecay
from itertools import combinations
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model
from sklearn.decomposition import PCA
# from deap import base, creator, tools, algorithms


In [ ]:
#upload the dataset
x_train = np.load('E://datasets//HAM10000//ARTIFACT_FREE//SPLIT//X_train.npy')
x_test = np.load('E://datasets//HAM10000//ARTIFACT_FREE//SPLIT//X_test.npy')
y_train = np.load('E://datasets//HAM10000//ARTIFACT_FREE//SPLIT//y_train.npy')
y_test = np.load('E://datasets//HAM10000//ARTIFACT_FREE//SPLIT//y_test.npy')

In [ ]:
num_classes = y_train.shape[1]

In [ ]:


# Load train and test predictions from all 6 models
model_paths = [
    "E://datasets//HAM10000//gnet//x_train_pred.npy", "E://datasets//HAM10000//gnet//x_test_pred.npy",
    "E://datasets//HAM10000//dnet//x_train_pred.npy", "E://datasets//HAM10000//dnet//x_test_pred.npy",
    "E://datasets//HAM10000//mnet//x_train_pred.npy", "E://datasets//HAM10000//mnet//x_test_pred.npy",
    "E://datasets//HAM10000//nnet//x_train_pred.npy", "E://datasets//HAM10000//nnet//x_test_pred.npy",
    "E://datasets//HAM10000//irnet//x_train_pred.npy", "E://datasets//HAM10000//irnet//x_test_pred.npy",
    "E://datasets//HAM10000//ccnngru//x_train_pred.npy", "E://datasets//HAM10000//ccnngru//x_test_pred.npy" ]

models = [np.load(path) for path in model_paths]

# Split into train & test sets
train_predictions = [models[i] for i in range(0, len(models), 2)]
test_predictions = [models[i] for i in range(1, len(models), 2)]

# Find best combination of models
best_acc = 0
best_combination = None
best_svm = None

# Try all combinations of models (1 to 6 models combined)
for r in range(1, 7):
    for combo in combinations(range(6), r):
        # Stack selected models
        X_train = np.hstack([train_predictions[i].reshape(-1, 1) for i in combo])
        # print("\t X_train",X_train.shape)
        X_test = np.hstack([test_predictions[i].reshape(-1, 1) for i in combo])
        # print("\t X_test",X_test.shape)

        # Train SVM
        svm = SVC(kernel='linear', probability=True, decision_function_shape='ovr')  # One-vs-Rest (OvR) for multiclass
        svm.fit(X_train, y_test)

        # Evaluate accuracy
        y_pred = svm.predict(X_test)
        acc = accuracy_score(y_test, y_pred)

        print(f"Combination {combo}: Accuracy = {acc:.4f}")

        # Save the best model
        if acc > best_acc:
            best_acc = acc
            best_combination = combo
            best_svm = svm

print(f"\nBest Combination: {best_combination} with Accuracy = {best_acc:.4f}")


# Evaluate Best Model
X_test_best = np.hstack([test_predictions[i].reshape(-1, 1) for i in best_combination])

# Ensure 2D shape
if X_test_best.ndim == 1:
    X_test_best = X_test_best.reshape(-1, 1)  # Fix single-model issue

y_pred_best = best_svm.predict(X_test_best)
np.save("E://datasets//HAM10000//svm//x_test_pred.npy",y_pred_best)
y_prob_best = best_svm.predict_proba(X_test_best)
np.save("E://datasets//HAM10000//svm//x_test_prob.npy",y_prob_best)


# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_best))

# Confusion Matrix
conf_matrix = confusion_matrix(y_test, y_pred_best)
print("\nConfusion Matrix:")
print(conf_matrix)

# Plot Confusion Matrix
# Compute confusion matrix (if not already computed)
conf_matrix = confusion_matrix(y_test, y_pred_best)
n_classes=num_classes
# Plot Confusion Matrix with Values
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", linewidths=0.5, cbar=True)
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Confusion Matrix')
plt.show()

# Compute ROC AUC for Multiclass (One-vs-Rest)
classes = np.unique(y_test)  # Ensures classes are an array-like
# Binarize y_test for ROC AUC computation
y_test_bin = label_binarize(y_test, classes=classes)  # Binarize labels for multiclass AUC
roc_auc = roc_auc_score(y_test_bin, y_prob_best, multi_class='ovr')
print(f"\nMulticlass ROC AUC Score: {roc_auc:.4f}")

# Plot ROC Curve for each class
plt.figure(figsize=(8, 6))

for i in range(n_classes):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_prob_best[:, i])
    plt.plot(fpr, tpr, label=f"Class {classes[i]} (AUC = {roc_auc_score(y_test_bin[:, i], y_prob_best[:, i]):.4f})")

plt.plot([0, 1], [0, 1], color='gray', linestyle='-')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Multiclass ROC Curve")
plt.legend()
plt.grid()
plt.show()
